# TaskMate — Validation Notebook 06: API Testing (FastAPI Layer)

**Purpose:** Verify the FastAPI application layer routes (`GET /health` and `POST /api/chat`), Firebase token authentication dependency, request validation, and response models using FastAPI `TestClient`.

**Operational Note:** This notebook is strictly for isolated experimentation, learning, and verification. It is **NOT** the production runtime.

## 1. Setup Environment and Imports

In [ ]:
import os
import sys
from unittest.mock import MagicMock, patch

# Ensure project root is in sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from fastapi.testclient import TestClient
from backend.app.main import app
from backend.app.api.dependencies import get_current_user
from backend.app.api.routes_chat import get_agent
from backend.app.agent.agent import AgentResponse

client = TestClient(app)
print("✅ FastAPI TestClient initialized successfully!")

## 2. Test Public Health Endpoint (GET /health)

In [ ]:
response = client.get("/health")
print("Status:", response.status_code)
print("Body:", response.json())
assert response.status_code == 200
assert response.json()["status"] == "ok"
print("✅ Health endpoint verified!")

## 3. Test Authentication Boundary (POST /api/chat)

In [ ]:
# Missing Authorization header
resp_no_auth = client.post("/api/chat", json={"message": "Hello"})
print("No Auth Status:", resp_no_auth.status_code)
assert resp_no_auth.status_code == 401

# Invalid token scheme / format
resp_bad_auth = client.post("/api/chat", json={"message": "Hello"}, headers={"Authorization": "InvalidToken"})
print("Bad Auth Status:", resp_bad_auth.status_code)
assert resp_bad_auth.status_code == 401

print("✅ Authentication rejection verified!")

## 4. Test Request Validation (POST /api/chat)

In [ ]:
# Empty whitespace message with auth override
app.dependency_overrides[get_current_user] = lambda: "test_user_colab"
try:
    resp_empty = client.post("/api/chat", json={"message": "   "})
    print("Empty Message Status:", resp_empty.status_code)
    assert resp_empty.status_code == 422
    print("✅ Empty prompt validation verified!")
finally:
    app.dependency_overrides.clear()

## 5. Test Authenticated Chat Turn (POST /api/chat)

In [ ]:
mock_agent = MagicMock()
mock_agent.process_message.return_value = AgentResponse(
    success=True,
    response="Hello! I am TaskMate, your task assistant.",
    tool_calls=[],
    tool_results=[],
    messages=[
        {"role": "user", "content": "Hello!"},
        {"role": "assistant", "content": "Hello! I am TaskMate, your task assistant."}
    ]
)

app.dependency_overrides[get_current_user] = lambda: "test_user_colab"
app.dependency_overrides[get_agent] = lambda: mock_agent

try:
    resp_chat = client.post("/api/chat", json={"message": "Hello!"})
    print("Chat Status:", resp_chat.status_code)
    print("Chat Response:", resp_chat.json())
    assert resp_chat.status_code == 200
    assert resp_chat.json()["success"] is True
    assert "TaskMate" in resp_chat.json()["response"]
    print("✅ Authenticated chat turn verified!")
finally:
    app.dependency_overrides.clear()